# Chapter 09 - Serverless Deep Learning Homework

## Overview
In this homework, we'll deploy the Straight vs Curly Hair Type model to serverless infrastructure.

### Model URL
https://github.com/alexeygrigorev/large-datasets/releases/download/hairstyle/model_2024_hairstyle.keras

## Setup and Dependencies

In [ ]:
# Install required packages
# !pip install tensorflow pillow numpy

In [ ]:
import tensorflow as tf
from tensorflow import keras
import numpy as np
from io import BytesIO
from urllib import request
from PIL import Image
import os

## Download the Model

In [ ]:
# Download the Keras model
model_url = "https://github.com/alexeygrigorev/large-datasets/releases/download/hairstyle/model_2024_hairstyle.keras"
model_path = "model_2024_hairstyle.keras"

if not os.path.exists(model_path):
    print(f"Downloading model from {model_url}...")
    request.urlretrieve(model_url, model_path)
    print(f"Model downloaded to {model_path}")
else:
    print(f"Model already exists at {model_path}")

## Question 1: Convert Model to TF-Lite Format

Convert the Keras model to TF-Lite format and check the size.

Options:
* 27 Mb
* 43 Mb
* 77 Mb
* 127 Mb

In [ ]:
# Load the Keras model
model = keras.models.load_model(model_path)
print(f"Model loaded successfully")
print(f"Input shape: {model.input_shape}")
print(f"Output shape: {model.output_shape}")

In [ ]:
# Convert to TF-Lite
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()

# Save the TF-Lite model
tflite_model_path = "model_2024_hairstyle.tflite"
with open(tflite_model_path, 'wb') as f:
    f.write(tflite_model)

# Get the file size
file_size_bytes = os.path.getsize(tflite_model_path)
file_size_mb = file_size_bytes / (1024 * 1024)

print(f"\nTF-Lite model saved to {tflite_model_path}")
print(f"File size: {file_size_bytes:,} bytes")
print(f"File size: {file_size_mb:.2f} MB")
print(f"\n✓ ANSWER Q1: {round(file_size_mb)} Mb")

## Question 2: Output Index for TF-Lite Model

To use this model, we need to know the index of the input and output.

What's the output index for this model?

Options:
* 3
* 7
* 13
* 24

In [ ]:
# Load the TF-Lite model
interpreter = tf.lite.Interpreter(model_path=tflite_model_path)
interpreter.allocate_tensors()

# Get input and output details
input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

print("Input details:")
for detail in input_details:
    print(f"  Index: {detail['index']}")
    print(f"  Shape: {detail['shape']}")
    print(f"  Type: {detail['dtype']}")

print("\nOutput details:")
for detail in output_details:
    print(f"  Index: {detail['index']}")
    print(f"  Shape: {detail['shape']}")
    print(f"  Type: {detail['dtype']}")

output_index = output_details[0]['index']
print(f"\n✓ ANSWER Q2: {output_index}")

## Image Preparation Functions

Helper functions for downloading and preparing images.

In [ ]:
def download_image(url):
    """Download an image from a URL"""
    with request.urlopen(url) as resp:
        buffer = resp.read()
    stream = BytesIO(buffer)
    img = Image.open(stream)
    return img

def prepare_image(img, target_size):
    """Prepare image for model input"""
    if img.mode != 'RGB':
        img = img.convert('RGB')
    img = img.resize(target_size, Image.NEAREST)
    return img

## Question 3: First Pixel R Channel Value

Download and preprocess the test image, then check the value in the first pixel (R channel).

Test image URL: https://habrastorage.org/webt/yf/_d/ok/yf_dokzqy3vcritme8ggnzqlvwa.jpeg

Based on homework 8, the target size should be (200, 200) and preprocessing involves rescaling to [0, 1].

Options:
* 0.24
* 0.44
* 0.64
* 0.84

In [ ]:
# Download the test image
test_image_url = "https://habrastorage.org/webt/yf/_d/ok/yf_dokzqy3vcritme8ggnzqlvwa.jpeg"
img = download_image(test_image_url)
print(f"Original image size: {img.size}")
print(f"Original image mode: {img.mode}")

# Display image if in notebook environment
try:
    display(img)
except:
    pass

In [ ]:
# Prepare the image with target size (200, 200) from homework 8
target_size = (200, 200)
img_prepared = prepare_image(img, target_size)
print(f"Prepared image size: {img_prepared.size}")

# Convert to numpy array
x = np.array(img_prepared, dtype='float32')
print(f"Array shape: {x.shape}")
print(f"Array dtype: {x.dtype}")

# Preprocess: rescale to [0, 1] (as done in homework 8)
x = x / 255.0

# Get the first pixel, R channel value
first_pixel_r = x[0, 0, 0]
print(f"\nFirst pixel RGB values: {x[0, 0, :]}")
print(f"First pixel R channel: {first_pixel_r:.2f}")
print(f"\n✓ ANSWER Q3: {first_pixel_r:.2f}")

## Question 4: Model Output for Test Image

Apply the TF-Lite model to the preprocessed image.

Options:
* 0.293
* 0.493
* 0.693
* 0.893

In [ ]:
# Add batch dimension
X = np.expand_dims(x, axis=0)
print(f"Input shape with batch dimension: {X.shape}")

# Set the input tensor
input_index = input_details[0]['index']
interpreter.set_tensor(input_index, X)

# Run inference
interpreter.invoke()

# Get the output
output_index = output_details[0]['index']
preds = interpreter.get_tensor(output_index)

print(f"Output shape: {preds.shape}")
print(f"Prediction value: {preds[0][0]:.3f}")
print(f"\n✓ ANSWER Q4: {preds[0][0]:.3f}")

## Question 5: Docker Base Image Size

Download the base Docker image and check its size.

Base image: `agrigorev/model-2024-hairstyle:v3`

Options:
* 182 Mb
* 382 Mb
* 582 Mb
* 782 Mb

**Note:** Run this in a terminal with Docker installed:
```bash
docker pull agrigorev/model-2024-hairstyle:v3
docker images agrigorev/model-2024-hairstyle:v3
```

In [ ]:
# This cell is for reference - execute in terminal
# Uncomment and run if Docker is available

# !docker pull agrigorev/model-2024-hairstyle:v3
# !docker images agrigorev/model-2024-hairstyle:v3

## Question 6: Extended Docker Container Output

Build and run the extended Docker container with Lambda code.

The container should:
- Use base image `agrigorev/model-2024-hairstyle:v3`
- Install TFLite runtime from: https://github.com/alexeygrigorev/tflite-aws-lambda/raw/main/tflite/tflite_runtime-2.14.0-cp310-cp310-linux_x86_64.whl
- Include lambda_function.py with preprocessing and inference code
- Use model file `model_2024_hairstyle_v2.tflite` (already in base image)

Test with same image URL and report the output.

Options:
* 0.229
* 0.429
* 0.629
* 0.829

**Note:** See `Dockerfile` and `lambda_function.py` in this directory for the implementation.

**Commands to run:**
```bash
# Build the Docker image
docker build -t hairstyle-lambda .

# Run the container
docker run -p 8080:8080 hairstyle-lambda

# Test with curl (in another terminal)
curl -X POST http://localhost:8080/2015-03-31/functions/function/invocations \
  -H 'Content-Type: application/json' \
  -d '{"url": "https://habrastorage.org/webt/yf/_d/ok/yf_dokzqy3vcritme8ggnzqlvwa.jpeg"}'
```

## Summary

This notebook walks through all the questions in the serverless homework:

1. **Q1**: Convert Keras model to TF-Lite and check file size
2. **Q2**: Find the output index of the TF-Lite model
3. **Q3**: Preprocess test image and get first pixel R channel value
4. **Q4**: Run inference on test image and get output
5. **Q5**: Check Docker base image size
6. **Q6**: Build and test Lambda container with TF-Lite

See the accompanying Python scripts for modular implementation:
- `q1_convert_model.py` - Model conversion
- `q2_model_info.py` - Model inspection
- `q3_preprocess_image.py` - Image preprocessing
- `q4_inference.py` - Model inference
- `lambda_function.py` - Lambda handler for Q6
- `Dockerfile` - Docker image for Q6